In [ ]:
# Imports

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dataset Path

dataset_path = "/content/drive/MyDrive/Final_Dataset"

print(os.listdir(dataset_path))

DATA PREPROCESSING

In [ ]:
 # Data Generator (rescale + validation split)

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [ ]:
# Training Data

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)


In [ ]:
# Validation Data

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
# Check classes

print("Classes:", train_data.class_indices)

 MODEL 1 = CNN (Convolutional Neural Network)

In [ ]:
# Build CNN Model

model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
])

In [ ]:
# Complie Model

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train Model

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,

)

In [ ]:
# Accuracy Graph

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.title("Accuracy Graph")
plt.show()

MODEL 2  = MobileNetV2


In [ ]:
# MobileNetV2

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

# Freeze layers
base_model.trainable = False

model2 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model2.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

MODEL 3 = ResNet50


In [ ]:
# ResNet50

base_model = tf.keras.applications.ResNet50(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model3 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history3 = model3.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

COMMPARISION GRAPH

In [ ]:
#  Comparision Graph

plt.plot(history.history['val_accuracy'], label='CNN')
plt.plot(history2.history['val_accuracy'], label='MobileNetV2')
plt.plot(history3.history['val_accuracy'], label='ResNet50')

plt.legend()
plt.title("Model Comparison")
plt.show()

UPLOAD IMAGE

In [ ]:
# Image Upload

from google.colab import files
uploaded = files.upload()

In [ ]:
print(uploaded.keys())

PREDICTION

In [ ]:
# Prediction (Test Images )

from tensorflow.keras.preprocessing import image
import numpy as np

#  AUTOMATIC FILE PICK
file_name = list(uploaded.keys())[0]

print("Using file:", file_name)

# IMAGE LOAD
img = image.load_img(file_name, target_size=(224,224))

# PREPROCESS
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

# PREDICT
pred = model.predict(img_array)

# CLASS NAME
class_names = list(train_data.class_indices.keys())
pred_class = class_names[np.argmax(pred)]

print("Prediction:", pred_class)

In [ ]:
# prediction if more than 1 image select

# for file_name in uploaded.keys():
#     print("\nImage:", file_name)

#     img = image.load_img(file_name, target_size=(224,224))
#     img_array = image.img_to_array(img)/255.0
#     img_array = np.expand_dims(img_array, axis=0)

#     pred = model.predict(img_array)
#     pred_class = class_names[np.argmax(pred)]

#     print("Prediction:", pred_class)